# 02 · Cleaning and customer snapshots
Rules CR-00…CR-12 (SPEC §4); snapshot design SPEC §5.

In [1]:
import os, pathlib
os.chdir(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd())
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import Image, display
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
from retail_targeting.config import load_config
cfg = load_config()
T = lambda name: pd.read_csv(f"outputs/tables/{name}")

In [2]:
pd.read_csv('outputs/reports/cleaning_log.csv')

,rule_id,description,rows_before,rows_after,rows_affected,value_affected,customers_affected
0,CR-07,exact duplicate rows removed,1044848,1033036,11812,54228.42,1594
1,CR-08,missing/invalid invoice date -> excluded/inval...,1033036,1033036,0,0.00,0
2,CR-05,bad-debt adjustment invoice (prefix A) -> excl...,1033036,1033030,6,-147614.08,0
3,CR-05b,"non-product stock code (postage, fees, manual,...",1033030,1027235,5795,75748.71,1119
4,CR-01/02,cancellation invoice with positive quantity ->...,1027235,1027235,0,0.00,0
5,CR-01/02,cancellation/return invoice (prefix C) -> adju...,1027235,1009320,17915,-716462.57,2445
6,CR-04,negative quantity without cancellation (stock ...,1009320,1005929,3391,0.00,0
7,CR-05c,price <= 0 -> excluded/zero_price,1005929,1003357,2572,0.00,43
8,CR-06,missing customer id -> excluded/missing_customer,1003357,776596,226761,2575278.90,0
9,CR-10,valid purchase line -> purchase,776596,0,776596,17068582.74,5852


In [3]:
pd.read_csv('outputs/reports/reconciliation.csv')

,check,expected,actual,abs_diff,status
0,raw value (after CR-07) == sum over line types,1.885553e+07,1.885553e+07,0.000000e+00,PASS
1,orders value == purchase+adjustment lines of k...,1.635859e+07,1.635859e+07,1.862645e-09,PASS
2,rows: raw after CR-07 == lines,1.033036e+06,1.033036e+06,0.000000e+00,PASS


In [4]:
T('snapshot_summary.csv')

,decision_date,split,eligible,prevalence,median_aov
0,2010-06-01,train,2652,0.500377,288.970000
1,2010-07-01,train,2737,0.509317,286.070000
2,2010-08-01,train,2813,0.570210,279.250000
3,2010-09-01,train,2822,0.620128,279.896111
4,2010-10-01,train,2899,0.579510,283.450000
5,2010-11-01,train,3196,0.493429,290.835000
6,2010-12-01,train,3465,0.380375,291.340000
7,2011-01-01,train,3403,0.377314,287.250000
8,2011-02-01,purged,3348,0.396655,291.502500
9,2011-03-01,purged,3338,0.423607,292.565000


Each row of `customer_snapshots` is (customer, T0) with features from [T0−180d, T0) and the target from [T0, T0+90d). Purged T0s separate train / validation / test by ≥ 90 days.